# Qwen3.5-9B 640px — 전체 Gold 최종 Refit

이 노트북은 모델 선택용 validation을 다시 사용하지 않습니다. 이미 확정한 640px 설정으로
전체 gold 5,073개와 고신뢰 dev pseudo-label을 사용해 fresh LoRA를 학습합니다.

- A100 80GB
- batch 16 자동 안전 검사
- 기존 best와 같은 총 학습량: batch 16 기준 684 optimizer update
- 전체 데이터에서는 약 1.8 epoch
- 200 update마다 Drive resume checkpoint
- 완료 후 640 test, 768 test, 640/768 17.5% blend를 모두 저장

런타임을 새로 시작한 뒤 **모두 실행**만 누르세요. 예상 시간은 약 2~2.5시간입니다.


## 1. 충돌 없는 환경 설치


In [ ]:
import sys, subprocess, importlib
import importlib.metadata as metadata

TARGET_VERSIONS = {"transformers": "5.15.1", "peft": "0.20.0", "Pillow": "11.3.0"}

def installed_version(package):
    try:
        return metadata.version(package)
    except metadata.PackageNotFoundError:
        return None

torchao_version = installed_version("torchao")
if torchao_version is not None:
    assert "torchao" not in sys.modules, (
        f"torchao {torchao_version}가 이미 로드되었습니다. 런타임을 삭제하고 다시 실행하세요."
    )
    print(f"사용하지 않는 충돌 패키지 torchao {torchao_version} 제거 중...")
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=True)
    importlib.invalidate_caches()
assert installed_version("torchao") is None

before = {package: installed_version(package) for package in TARGET_VERSIONS}
ready = all(before[package] == version for package, version in TARGET_VERSIONS.items())
if not ready:
    preloaded = [name for name in ("transformers", "peft", "PIL") if name in sys.modules]
    if preloaded:
        raise RuntimeError(
            f"설치 전 이미 import된 패키지가 있습니다: {preloaded}. "
            "런타임 > 연결 해제 및 런타임 삭제 후 다시 모두 실행하세요."
        )
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir", "--upgrade",
        "transformers==5.15.1", "peft==0.20.0", "accelerate>=1.14.0",
        "safetensors>=0.6.0", "Pillow==11.3.0", "pandas>=2.2", "tqdm>=4.66",
    ], check=True)

after = {package: installed_version(package) for package in TARGET_VERSIONS}
assert after == TARGET_VERSIONS, f"패키지 버전 불일치: {after}"
print("환경 준비 완료:", after)


## 2. 최종 refit 설정


In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import gc, json, math, random, shutil, zipfile
from collections import Counter
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image, ImageOps
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

# 기존 seed 42와 독립적인 최종 모델을 얻습니다.
SEED = 43
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision("high")
Image.MAX_IMAGE_PIXELS = None

assert torch.cuda.is_available(), "GPU 런타임이 필요합니다."
DEVICE = torch.device("cuda:0")
GPU_NAME = torch.cuda.get_device_name(0)
GPU_VRAM_GIB = torch.cuda.get_device_properties(0).total_memory / 1024**3
assert GPU_VRAM_GIB >= 70, f"A100 80GB가 필요합니다: {GPU_NAME}, {GPU_VRAM_GIB:.1f} GiB"

MODEL_ID = "Qwen/Qwen3.5-9B"
DATA_ROOT = Path("/content")
DATA_ARCHIVE = Path("/content/drive/MyDrive/2026-ssafy-15-2-ai.zip")
IMAGE_SIZE = 640
TRAIN_BATCH_CANDIDATES = [16, 12, 8, 4]
MEMORY_SAFETY_RATIO = 0.88
INFER_BATCH_SIZE = 8
NUM_WORKERS = 0

REFERENCE_BATCH_SIZE = 16
REFERENCE_TOTAL_UPDATES = 684
# 기존 640 run은 3 epoch=1,026 update cosine schedule에서 update 684가 best였습니다.
# scheduler horizon은 그대로 두고 학습만 684에서 멈춰 당시 LR 궤적을 보존합니다.
REFERENCE_SCHEDULER_UPDATES = 1026
SAVE_EVERY_UPDATES = 200
BASE_BATCH_SIZE = 8
BASE_LEARNING_RATE = 5e-5
MAX_LEARNING_RATE = 7e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.03
MAX_GRAD_NORM = 1.0
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
MIN_DEV_VOTES = 4
PSEUDO_CAP_PER_CATEGORY = 0.75
AUGMENT_HORIZONTAL_FLIP = True
EXPORT_TO_DRIVE = True
AUTO_RELEASE_RUNTIME = True
RELEASE_DELAY_SECONDS = 20

RUN_STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_ROOT = Path("/content/qwen35_9b_full_refit") / RUN_STAMP
FINAL_ADAPTER_DIR = OUTPUT_ROOT / "final_adapter"
DRIVE_OUTPUT_ROOT = Path("/content/drive/MyDrive/qwen35_9b_full_refit")
DRIVE_RESUME_ROOT = DRIVE_OUTPUT_ROOT / "resume_seed43_u684"
RESUME_ADAPTER_DIR = DRIVE_RESUME_ROOT / "adapter"
RESUME_STATE_PATH = DRIVE_RESUME_ROOT / "training_state.pt"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

TEST640_PATH = OUTPUT_ROOT / "qwen35_full_refit_640_test.csv"
TEST768_PATH = OUTPUT_ROOT / "qwen35_full_refit_768_test.csv"
BLEND_PATH = OUTPUT_ROOT / "qwen35_full_refit_640_768_w0175_test.csv"

LETTERS = ["a", "b", "c", "d"]
LETTER_TO_INDEX = {letter: index for index, letter in enumerate(LETTERS)}
PROB_COLUMNS = [f"prob_{letter}" for letter in LETTERS]

def clear_cuda():
    gc.collect()
    torch.cuda.empty_cache()

def cuda_memory():
    return {
        "allocated_GB": round(torch.cuda.memory_allocated() / 1e9, 2),
        "reserved_GB": round(torch.cuda.memory_reserved() / 1e9, 2),
        "peak_GB": round(torch.cuda.max_memory_allocated() / 1e9, 2),
    }

print("GPU:", GPU_NAME, f"{GPU_VRAM_GIB:.1f} GiB")
print("full-data fresh refit / output:", OUTPUT_ROOT)


## 3. Drive 마운트와 전체 데이터 준비


In [ ]:
from google.colab import drive
if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

required = [
    DATA_ROOT / "train.csv", DATA_ROOT / "dev.csv", DATA_ROOT / "test.csv",
    DATA_ROOT / "train", DATA_ROOT / "dev", DATA_ROOT / "test",
]
if not all(path.exists() for path in required):
    assert DATA_ARCHIVE.exists(), f"데이터 ZIP이 없습니다: {DATA_ARCHIVE}"
    with zipfile.ZipFile(DATA_ARCHIVE) as archive:
        archive.extractall(DATA_ROOT)
missing = [str(path) for path in required if not path.exists()]
assert not missing, f"필수 데이터 누락: {missing}"

train_df = pd.read_csv(DATA_ROOT / "train.csv")
dev_df = pd.read_csv(DATA_ROOT / "dev.csv")
test_df = pd.read_csv(DATA_ROOT / "test.csv")

def categorize(question):
    question = str(question)
    if "몇 개" in question or "개수" in question:
        return "counting"
    if "재질" in question or "소재" in question:
        return "material"
    if "색" in question:
        return "color"
    if "종류" in question:
        return "type"
    return "other"

for frame in (train_df, dev_df, test_df):
    frame["category"] = frame["question"].map(categorize)

assert len(train_df) == 5073 and len(test_df) == 5074
assert train_df["answer"].isin(LETTERS).all()
assert train_df["id"].is_unique and dev_df["id"].is_unique and test_df["id"].is_unique

# 핵심 변경: validation 508개를 제외하지 않고 gold 전체를 학습에 사용합니다.
gold_train_df = train_df.copy().reset_index(drop=True)

def image_path(relative_path):
    path = Path(str(relative_path))
    return path if path.is_absolute() else DATA_ROOT / path

for frame in (gold_train_df, test_df):
    missing_images = [str(x) for x in frame["path"].head(100) if not image_path(x).exists()]
    assert not missing_images, f"이미지 경로 실패: {missing_images[:3]}"

print("full gold:", len(gold_train_df), "/ dev:", len(dev_df), "/ test:", len(test_df))
print("gold category\n", gold_train_df["category"].value_counts())


## 4. 프롬프트와 추론 함수


In [ ]:
SYSTEM_INSTRUCTION = (
    "You are an expert visual multiple-choice question answering system. "
    "Inspect the entire image carefully. For quantity questions, count every relevant visible object exactly once. "
    "Answer with exactly one lowercase letter: a, b, c, or d. Do not explain."
)

def build_prompt(row):
    return (
        f"{row['question']}\n"
        f"(a) {row['a']}\n(b) {row['b']}\n(c) {row['c']}\n(d) {row['d']}\n\n"
        "정답을 a, b, c, d 중 한 글자로만 출력하세요."
    )

def build_messages(row, image, answer=None):
    messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_INSTRUCTION}]},
        {"role": "user", "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": build_prompt(row)},
        ]},
    ]
    if answer is not None:
        messages.append({"role": "assistant", "content": [{"type": "text", "text": str(answer)}]})
    return messages

def apply_template(messages, add_generation_prompt):
    kwargs = dict(tokenize=False, add_generation_prompt=add_generation_prompt)
    try:
        return processor.apply_chat_template(messages, enable_thinking=False, **kwargs)
    except TypeError:
        return processor.apply_chat_template(messages, **kwargs)

def evaluate_predictions(frame, title):
    correct = int((frame["answer"] == frame["pred"]).sum())
    print(f"\n=== {title} ===")
    print(f"전체: {correct}/{len(frame)} = {correct/len(frame):.4f}")
    print(frame.groupby("category")["correct"].agg(["mean", "sum", "count"]))
    print("pred distribution:", frame["pred"].value_counts().sort_index().to_dict())
    return correct

def prediction_nll(frame):
    matrix = frame[PROB_COLUMNS].to_numpy(dtype=np.float64)
    indices = frame["answer"].map(LETTER_TO_INDEX).to_numpy()
    gold_probability = matrix[np.arange(len(frame)), indices]
    return float(-np.log(np.clip(gold_probability, 1e-12, 1.0)).mean())


## 5. Qwen3.5-9B base와 640 processor 로드


In [ ]:
from transformers import AutoProcessor
try:
    from transformers import Qwen3_5ForConditionalGeneration as Qwen35Model
except ImportError:
    from transformers import AutoModelForMultimodalLM as Qwen35Model

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=IMAGE_SIZE * IMAGE_SIZE,
    max_pixels=IMAGE_SIZE * IMAGE_SIZE,
    trust_remote_code=True,
)
if processor.tokenizer.pad_token_id is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token
processor.tokenizer.padding_side = "left"

model = Qwen35Model.from_pretrained(
    MODEL_ID,
    dtype=torch.bfloat16,
    device_map={"": 0},
    low_cpu_mem_usage=True,
    trust_remote_code=True,
    attn_implementation="sdpa",
)
model.eval()
MODEL_DEVICE = next(model.parameters()).device

dummy_messages = [
    {"role": "system", "content": [{"type": "text", "text": SYSTEM_INSTRUCTION}]},
    {"role": "user", "content": [{"type": "text", "text": "Choose one: (a) A (b) B (c) C (d) D"}]},
]
dummy_prefix = apply_template(dummy_messages, add_generation_prompt=True)
dummy_ids = processor.tokenizer(dummy_prefix, add_special_tokens=False)["input_ids"]
LETTER_TOKEN_IDS = {}
for letter in LETTERS:
    extended = processor.tokenizer(dummy_prefix + letter, add_special_tokens=False)["input_ids"]
    assert extended[:len(dummy_ids)] == dummy_ids and len(extended) > len(dummy_ids)
    LETTER_TOKEN_IDS[letter] = extended[len(dummy_ids)]
assert len(set(LETTER_TOKEN_IDS.values())) == 4
print("letter token ids:", LETTER_TOKEN_IDS)

def forward_last_logits(active_model, inputs):
    try:
        return active_model(**inputs, use_cache=False, logits_to_keep=1)
    except TypeError:
        return active_model(**inputs, use_cache=False)

def score_letters(active_model, dataframe, desc):
    old_side = processor.tokenizer.padding_side
    processor.tokenizer.padding_side = "left"
    active_model.eval()
    probabilities = []
    letter_tensor = torch.tensor([LETTER_TOKEN_IDS[x] for x in LETTERS], device=MODEL_DEVICE)
    with torch.inference_mode():
        for start in tqdm(range(0, len(dataframe), INFER_BATCH_SIZE), desc=desc, unit="batch"):
            chunk = dataframe.iloc[start:start + INFER_BATCH_SIZE]
            images, texts = [], []
            for _, row in chunk.iterrows():
                with Image.open(image_path(row["path"])) as opened:
                    image = ImageOps.exif_transpose(opened).convert("RGB")
                images.append(image)
                texts.append(apply_template(build_messages(row, image), add_generation_prompt=True))
            inputs = processor(text=texts, images=images, padding=True, return_tensors="pt").to(MODEL_DEVICE)
            with torch.autocast("cuda", dtype=torch.bfloat16):
                outputs = forward_last_logits(active_model, inputs)
            logits = outputs.logits[:, -1, :].index_select(-1, letter_tensor)
            probs = torch.softmax(logits.float(), dim=-1)
            probabilities.extend(probs.cpu().tolist())
            del inputs, outputs, logits, probs, images, texts
    processor.tokenizer.padding_side = old_side
    result = dataframe.reset_index(drop=True).copy()
    for index, letter in enumerate(LETTERS):
        result[f"prob_{letter}"] = [row[index] for row in probabilities]
    result["pred"] = [LETTERS[int(np.argmax(row))] for row in probabilities]
    result["pred_conf"] = [float(max(row)) for row in probabilities]
    if "answer" in result:
        result["correct"] = result["pred"] == result["answer"]
    return result

print("base loaded:", MODEL_DEVICE, cuda_memory())


## 6. Fresh Vision + Language LoRA


In [ ]:
from peft import LoraConfig, get_peft_model

LANGUAGE_LEAVES = {
    "q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj",
    "in_proj_qkv", "in_proj_z", "out_proj",
}
VISION_LEAVES = {"qkv", "proj", "linear_fc1", "linear_fc2", "gate_proj", "up_proj", "down_proj"}

visual_targets, language_targets = [], []
for name, module in model.named_modules():
    if not isinstance(module, nn.Linear):
        continue
    leaf = name.rsplit(".", 1)[-1]
    padded = f".{name}."
    if ".visual." in padded and leaf in VISION_LEAVES:
        visual_targets.append(name)
    elif ".language_model.layers." in padded and leaf in LANGUAGE_LEAVES:
        language_targets.append(name)

target_modules = visual_targets + language_targets
assert visual_targets and language_targets
print("vision targets:", len(visual_targets), visual_targets[:10])
print("language targets:", len(language_targets), language_targets[:10])

lora_config = LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    bias="none", target_modules=target_modules, task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model.enable_input_require_grads()
if hasattr(model.config, "use_cache"):
    model.config.use_cache = False
if hasattr(model.config, "text_config"):
    model.config.text_config.use_cache = False
model.print_trainable_parameters()
MODEL_DEVICE = next(model.parameters()).device
print("fresh LoRA attached / memory:", cuda_memory())


## 7. 전체 Gold + 고신뢰 pseudo-label 학습셋


In [ ]:
VOTE_COLUMNS = ["answer1", "answer2", "answer3", "answer4", "answer5"]

def majority_vote(row):
    votes = [
        str(row[column]).strip().lower()
        for column in VOTE_COLUMNS
        if pd.notna(row[column]) and str(row[column]).strip().lower() in LETTERS
    ]
    if not votes:
        return pd.Series({"answer": None, "vote_count": 0, "vote_margin": 0})
    ordered = sorted(Counter(votes).items(), key=lambda item: (-item[1], item[0]))
    answer, count = ordered[0]
    second = ordered[1][1] if len(ordered) > 1 else 0
    return pd.Series({"answer": answer, "vote_count": count, "vote_margin": count - second})

vote_result = dev_df.apply(majority_vote, axis=1)
dev_labeled = dev_df.drop(columns=VOTE_COLUMNS).copy()
dev_labeled[["answer", "vote_count", "vote_margin"]] = vote_result
dev_labeled = dev_labeled[
    (dev_labeled["vote_count"] >= MIN_DEV_VOTES) & dev_labeled["answer"].isin(LETTERS)
].copy()
dev_labeled["source"] = "dev_pseudo"
gold_train_df["source"] = "gold"
gold_train_df["vote_count"] = 99
gold_train_df["vote_margin"] = 99

selected_pseudo = []
for category, gold_group in gold_train_df.groupby("category"):
    candidates = dev_labeled[dev_labeled["category"] == category].copy()
    cap = int(math.ceil(len(gold_group) * PSEUDO_CAP_PER_CATEGORY))
    candidates = candidates.sort_values(
        ["vote_count", "vote_margin"], ascending=False, kind="stable"
    ).head(cap)
    selected_pseudo.append(candidates)
selected_pseudo_df = pd.concat(selected_pseudo, ignore_index=True)

finetune_df = pd.concat([gold_train_df, selected_pseudo_df], ignore_index=True, sort=False)
finetune_df = finetune_df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
assert len(gold_train_df) == 5073
print("gold:", len(gold_train_df), "/ pseudo:", len(selected_pseudo_df), "/ total:", len(finetune_df))
print(finetune_df.groupby(["category", "source"]).size())


## 8. Dataset과 batch 16 안전 검사


In [ ]:
class FullVQADataset(Dataset):
    def __init__(self, dataframe):
        self.dataframe = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]
        with Image.open(image_path(row["path"])) as opened:
            image = ImageOps.exif_transpose(opened).convert("RGB")
        horizontal_reference = any(
            term in str(row["question"]) for term in ["왼쪽", "오른쪽", "좌측", "우측"]
        )
        if AUGMENT_HORIZONTAL_FLIP and not horizontal_reference and random.random() < 0.5:
            image = ImageOps.mirror(image)
        return {"row": row, "image": image, "answer": str(row["answer"]).strip().lower()}

class LetterOnlyCollator:
    def __call__(self, items):
        images, texts, answers = [], [], []
        for item in items:
            row, image, answer = item["row"], item["image"], item["answer"]
            assert answer in LETTERS
            images.append(image)
            answers.append(answer)
            texts.append(apply_template(build_messages(row, image, answer), add_generation_prompt=False))

        old_side = processor.tokenizer.padding_side
        processor.tokenizer.padding_side = "right"
        encoded = processor(text=texts, images=images, padding=True, return_tensors="pt")
        labels = torch.full_like(encoded["input_ids"], -100)
        for index, answer in enumerate(answers):
            length = int(encoded["attention_mask"][index].sum())
            token_id = LETTER_TOKEN_IDS[answer]
            candidates = torch.where(encoded["input_ids"][index, :length] == token_id)[0]
            candidates = candidates[candidates >= max(0, length - 32)]
            assert len(candidates) >= 1, f"assistant answer token 없음: {answer}"
            labels[index, int(candidates[-1])] = token_id
        encoded["labels"] = labels
        processor.tokenizer.padding_side = old_side
        return encoded

train_dataset = FullVQADataset(finetune_df)

def make_train_loader(batch_size, shuffle=True):
    return DataLoader(
        train_dataset, batch_size=batch_size, shuffle=shuffle,
        num_workers=NUM_WORKERS, pin_memory=True, collate_fn=LetterOnlyCollator(),
    )

TRAIN_BATCH_SIZE = None
batch_trials = []
for candidate in TRAIN_BATCH_CANDIDATES:
    trial_loader = make_train_loader(candidate, shuffle=False)
    trial_batch = next(iter(trial_loader))
    try:
        clear_cuda()
        torch.cuda.reset_peak_memory_stats()
        model.train()
        model.zero_grad(set_to_none=True)
        trial_batch = {
            key: value.to(MODEL_DEVICE, non_blocking=True) if torch.is_tensor(value) else value
            for key, value in trial_batch.items()
        }
        with torch.autocast("cuda", dtype=torch.bfloat16):
            trial_output = model(**trial_batch, use_cache=False)
        trial_output.loss.backward()
        peak_gib = torch.cuda.max_memory_allocated() / 1024**3
        finite_gradients = [
            name for name, parameter in model.named_parameters()
            if parameter.requires_grad and parameter.grad is not None
            and torch.isfinite(parameter.grad).all() and float(parameter.grad.abs().max()) > 0
        ]
        assert any("visual" in name for name in finite_gradients)
        assert any("language_model" in name for name in finite_gradients)
        safe = peak_gib <= GPU_VRAM_GIB * MEMORY_SAFETY_RATIO
        batch_trials.append({"batch": candidate, "peak_GiB": round(peak_gib, 2), "safe": safe})
        print("batch trial:", batch_trials[-1])
        if safe:
            TRAIN_BATCH_SIZE = candidate
            break
    except (torch.OutOfMemoryError, RuntimeError) as error:
        if "out of memory" not in str(error).lower() and not isinstance(error, torch.OutOfMemoryError):
            raise
        batch_trials.append({"batch": candidate, "error": "OOM", "safe": False})
        print("batch trial OOM:", candidate)
    finally:
        model.zero_grad(set_to_none=True)
        for variable in ("trial_batch", "trial_output", "trial_loader"):
            if variable in globals():
                del globals()[variable]
        clear_cuda()

assert TRAIN_BATCH_SIZE is not None, f"안전한 batch를 찾지 못했습니다: {batch_trials}"
LEARNING_RATE = min(
    MAX_LEARNING_RATE,
    BASE_LEARNING_RATE * math.sqrt(TRAIN_BATCH_SIZE / BASE_BATCH_SIZE),
)
train_loader = make_train_loader(TRAIN_BATCH_SIZE, shuffle=True)
print("selected batch:", TRAIN_BATCH_SIZE, "/ lr:", LEARNING_RATE)
print("steps per epoch:", len(train_loader), "/ trials:", batch_trials)


## 9. 총 684-update 학습 · 자동 Drive resume


In [ ]:
from transformers import get_cosine_schedule_with_warmup
from peft import set_peft_model_state_dict
from safetensors.torch import load_file

# batch가 예상과 다르면 동일한 총 학습 이미지 수를 유지합니다.
MAX_UPDATES = int(math.ceil(
    REFERENCE_TOTAL_UPDATES * REFERENCE_BATCH_SIZE / TRAIN_BATCH_SIZE
))
SCHEDULER_TOTAL_UPDATES = int(math.ceil(
    REFERENCE_SCHEDULER_UPDATES * REFERENCE_BATCH_SIZE / TRAIN_BATCH_SIZE
))
TARGET_EXAMPLES = REFERENCE_TOTAL_UPDATES * REFERENCE_BATCH_SIZE
effective_epochs = TARGET_EXAMPLES / len(finetune_df)
print("fixed updates:", MAX_UPDATES, "/ target examples:", TARGET_EXAMPLES,
      "/ scheduler horizon:", SCHEDULER_TOTAL_UPDATES,
      "/ effective epochs:", round(effective_epochs, 3))

trainable_parameters = [parameter for parameter in model.parameters() if parameter.requires_grad]
optimizer = torch.optim.AdamW(
    trainable_parameters, lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY, betas=(0.9, 0.95)
)
warmup_updates = int(SCHEDULER_TOTAL_UPDATES * WARMUP_RATIO)
scheduler = get_cosine_schedule_with_warmup(
    optimizer, warmup_updates, SCHEDULER_TOTAL_UPDATES
)

global_update = 0
history = []

def save_resume_checkpoint():
    local_adapter = OUTPUT_ROOT / "resume_adapter"
    local_state = OUTPUT_ROOT / "training_state.pt"
    model.save_pretrained(local_adapter, safe_serialization=True)
    torch.save({
        "global_update": global_update,
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "history": history,
        "train_batch_size": TRAIN_BATCH_SIZE,
        "max_updates": MAX_UPDATES,
        "scheduler_total_updates": SCHEDULER_TOTAL_UPDATES,
    }, local_state)
    DRIVE_RESUME_ROOT.mkdir(parents=True, exist_ok=True)
    shutil.copytree(local_adapter, RESUME_ADAPTER_DIR, dirs_exist_ok=True)
    shutil.copy2(local_state, RESUME_STATE_PATH)
    print("resume checkpoint saved:", global_update, DRIVE_RESUME_ROOT)

if RESUME_STATE_PATH.exists() and (RESUME_ADAPTER_DIR / "adapter_model.safetensors").exists():
    adapter_state = load_file(str(RESUME_ADAPTER_DIR / "adapter_model.safetensors"), device="cpu")
    print("resume adapter:", set_peft_model_state_dict(model, adapter_state))
    del adapter_state
    saved = torch.load(RESUME_STATE_PATH, map_location="cpu", weights_only=False)
    assert int(saved["train_batch_size"]) == TRAIN_BATCH_SIZE
    assert int(saved["max_updates"]) == MAX_UPDATES
    assert int(saved["scheduler_total_updates"]) == SCHEDULER_TOTAL_UPDATES
    optimizer.load_state_dict(saved["optimizer"])
    scheduler.load_state_dict(saved["scheduler"])
    global_update = int(saved["global_update"])
    history = list(saved.get("history", []))
    del saved
    clear_cuda()
    print("resumed at update:", global_update)

model.train()
optimizer.zero_grad(set_to_none=True)
running_loss = 0.0
window_loss = 0.0
window_steps = 0
progress = tqdm(total=MAX_UPDATES, initial=global_update, desc="full-data refit 640")

while global_update < MAX_UPDATES:
    for batch in train_loader:
        if global_update >= MAX_UPDATES:
            break
        batch = {
            key: value.to(MODEL_DEVICE, non_blocking=True) if torch.is_tensor(value) else value
            for key, value in batch.items()
        }
        with torch.autocast("cuda", dtype=torch.bfloat16):
            outputs = model(**batch, use_cache=False)
            loss = outputs.loss
        loss.backward()
        loss_value = float(loss.detach())
        running_loss += loss_value
        window_loss += loss_value
        window_steps += 1
        torch.nn.utils.clip_grad_norm_(trainable_parameters, MAX_GRAD_NORM)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad(set_to_none=True)
        global_update += 1
        progress.update(1)

        if global_update % 25 == 0 or global_update == MAX_UPDATES:
            record = {
                "update": global_update,
                "mean_loss": running_loss / global_update,
                "window_loss": window_loss / max(window_steps, 1),
                "learning_rate": scheduler.get_last_lr()[0],
                "peak_GB": torch.cuda.max_memory_allocated() / 1e9,
            }
            history.append(record)
            pd.DataFrame(history).to_csv(OUTPUT_ROOT / "training_history.csv", index=False)
            progress.set_postfix(
                loss=f"{record['window_loss']:.4f}",
                lr=f"{record['learning_rate']:.2e}",
                mem=f"{record['peak_GB']:.1f}G",
            )
            window_loss = 0.0
            window_steps = 0

        if global_update % SAVE_EVERY_UPDATES == 0:
            save_resume_checkpoint()
        del batch, outputs, loss

progress.close()
assert global_update == MAX_UPDATES
model.save_pretrained(FINAL_ADAPTER_DIR, safe_serialization=True)
save_resume_checkpoint()
print("training complete:", global_update, "/ memory:", cuda_memory())

del optimizer, scheduler, trainable_parameters, train_loader, train_dataset
clear_cuda()


## 10. Sanity check와 640/768 test 추론


In [ ]:
model.eval()

# 학습 자체가 깨지지 않았는지만 확인하는 in-sample sanity check입니다.
sanity_df = train_df.sample(n=64, random_state=SEED).reset_index(drop=True)
sanity_scored = score_letters(model, sanity_df, "full refit in-sample sanity")
sanity_accuracy = float(sanity_scored["correct"].mean())
sanity_scored.to_csv(OUTPUT_ROOT / "refit_sanity_train64.csv", index=False)
print("sanity accuracy:", sanity_accuracy)
assert sanity_accuracy >= 0.75, "학습 결과가 비정상적이므로 test 제출 생성을 중단합니다."

test640 = score_letters(model, test_df, "full refit test 640")
test640.to_csv(TEST640_PATH, index=False)

from transformers import AutoProcessor
processor768 = AutoProcessor.from_pretrained(
    MODEL_ID, min_pixels=768 * 768, max_pixels=768 * 768, trust_remote_code=True,
)
if processor768.tokenizer.pad_token_id is None:
    processor768.tokenizer.pad_token = processor768.tokenizer.eos_token
processor768.tokenizer.padding_side = "left"

def apply_template_with(active_processor, messages):
    kwargs = dict(tokenize=False, add_generation_prompt=True)
    try:
        return active_processor.apply_chat_template(messages, enable_thinking=False, **kwargs)
    except TypeError:
        return active_processor.apply_chat_template(messages, **kwargs)

def score_letters_with_processor(active_model, dataframe, active_processor, desc):
    probabilities = []
    letter_tensor = torch.tensor([LETTER_TOKEN_IDS[x] for x in LETTERS], device=MODEL_DEVICE)
    active_model.eval()
    with torch.inference_mode():
        for start in tqdm(range(0, len(dataframe), INFER_BATCH_SIZE), desc=desc, unit="batch"):
            chunk = dataframe.iloc[start:start + INFER_BATCH_SIZE]
            images, texts = [], []
            for _, row in chunk.iterrows():
                with Image.open(image_path(row["path"])) as opened:
                    image = ImageOps.exif_transpose(opened).convert("RGB")
                images.append(image)
                texts.append(apply_template_with(active_processor, build_messages(row, image)))
            inputs = active_processor(
                text=texts, images=images, padding=True, return_tensors="pt"
            ).to(MODEL_DEVICE)
            with torch.autocast("cuda", dtype=torch.bfloat16):
                outputs = forward_last_logits(active_model, inputs)
            logits = outputs.logits[:, -1, :].index_select(-1, letter_tensor)
            batch_probabilities = torch.softmax(logits.float(), dim=-1)
            probabilities.extend(batch_probabilities.cpu().tolist())
            del inputs, outputs, logits, batch_probabilities, images, texts
    result = dataframe.reset_index(drop=True).copy()
    matrix = np.asarray(probabilities, dtype=np.float64)
    for index, letter in enumerate(LETTERS):
        result[f"prob_{letter}"] = matrix[:, index]
    result["pred"] = [LETTERS[index] for index in matrix.argmax(axis=1)]
    result["pred_conf"] = matrix.max(axis=1)
    return result

test768 = score_letters_with_processor(model, test_df, processor768, "full refit test 768")
test768.to_csv(TEST768_PATH, index=False)

def normalized_probabilities(frame):
    values = frame[PROB_COLUMNS].to_numpy(dtype=np.float64)
    return values / values.sum(axis=1, keepdims=True)

def log_blend(left, right, right_weight):
    score = (
        (1.0 - right_weight) * np.log(np.clip(left, 1e-12, 1.0))
        + right_weight * np.log(np.clip(right, 1e-12, 1.0))
    )
    score -= score.max(axis=1, keepdims=True)
    values = np.exp(score)
    return values / values.sum(axis=1, keepdims=True)

blend_probability = log_blend(
    normalized_probabilities(test640), normalized_probabilities(test768), 0.175
)
blend_test = test640.copy()
for index, letter in enumerate(LETTERS):
    blend_test[f"prob_{letter}"] = blend_probability[:, index]
blend_test["pred"] = [LETTERS[index] for index in blend_probability.argmax(axis=1)]
blend_test["pred_conf"] = blend_probability.max(axis=1)
blend_test.to_csv(BLEND_PATH, index=False)

def write_submission(scored, filename):
    submission = scored[["id", "pred"]].rename(columns={"pred": "answer"}).copy()
    assert len(submission) == 5074 and submission["id"].is_unique
    assert submission["answer"].isin(LETTERS).all()
    path = OUTPUT_ROOT / filename
    submission.to_csv(path, index=False)
    print(path, "/", submission["answer"].value_counts().to_dict())
    return path

SUBMISSION640 = write_submission(test640, "submission_full_refit_640.csv")
SUBMISSION768 = write_submission(test768, "submission_full_refit_768.csv")
SUBMISSION_BLEND = write_submission(
    blend_test, "submission_full_refit_640_768_w0175.csv"
)


## 11. 메타데이터와 Drive 최종 백업


In [ ]:
metadata_payload = {
    "model_id": MODEL_ID,
    "training_mode": "fresh_full_gold_fixed_updates",
    "image_size": IMAGE_SIZE,
    "seed": SEED,
    "selected_train_batch_size": TRAIN_BATCH_SIZE,
    "reference_batch_size": REFERENCE_BATCH_SIZE,
    "reference_total_updates": REFERENCE_TOTAL_UPDATES,
    "reference_scheduler_updates": REFERENCE_SCHEDULER_UPDATES,
    "actual_total_updates": MAX_UPDATES,
    "scheduler_total_updates": SCHEDULER_TOTAL_UPDATES,
    "target_examples": TARGET_EXAMPLES,
    "effective_epochs": effective_epochs,
    "learning_rate": LEARNING_RATE,
    "train_rows": len(finetune_df),
    "gold_rows": len(gold_train_df),
    "pseudo_rows": len(selected_pseudo_df),
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "sanity_accuracy": sanity_accuracy,
    "test_640_768_blend_weight_768": 0.175,
    "gpu": GPU_NAME,
    "gpu_vram_gib": GPU_VRAM_GIB,
}
with open(OUTPUT_ROOT / "run_metadata.json", "w", encoding="utf-8") as file:
    json.dump(metadata_payload, file, ensure_ascii=False, indent=2)

if EXPORT_TO_DRIVE:
    drive_run = DRIVE_OUTPUT_ROOT / RUN_STAMP
    drive_run.mkdir(parents=True, exist_ok=True)
    for artifact in OUTPUT_ROOT.iterdir():
        if artifact.is_file() and artifact.suffix in {".csv", ".json"}:
            shutil.copy2(artifact, drive_run / artifact.name)
    shutil.copytree(FINAL_ADAPTER_DIR, drive_run / "final_adapter", dirs_exist_ok=True)
    with open(drive_run / "COMPLETED.txt", "w", encoding="utf-8") as file:
        file.write(f"completed {RUN_STAMP}, updates={MAX_UPDATES}\n")
    print("Drive backup:", drive_run)

print("\n완료")
print("640 probabilities:", TEST640_PATH)
print("768 probabilities:", TEST768_PATH)
print("640/768 blend probabilities:", BLEND_PATH)
print("ready submissions:", SUBMISSION640, SUBMISSION768, SUBMISSION_BLEND)
print("local output:", OUTPUT_ROOT)
print("memory:", cuda_memory())


## 아침에 가져올 파일

Drive의 `/MyDrive/qwen35_9b_full_refit/<실행시각>/` 폴더를 통째로 내려받으세요.
특히 아래 파일이 필요합니다.

1. `qwen35_full_refit_640_test.csv`
2. `qwen35_full_refit_768_test.csv`
3. `qwen35_full_refit_640_768_w0175_test.csv`
4. `run_metadata.json`
5. `final_adapter/`

세 submission은 단독 진단용입니다. 가장 중요한 작업은 확률 CSV를 현재 0.94836
앙상블과 보수적으로 결합하는 것입니다.


## 12. 모든 저장 완료 후 GPU 런타임 자동 해제


In [ ]:
import sys, time

if AUTO_RELEASE_RUNTIME:
    assert EXPORT_TO_DRIVE, "자동 해제 전 Drive 백업이 반드시 활성화되어야 합니다."
    required_drive_artifacts = [
        drive_run / "COMPLETED.txt",
        drive_run / TEST640_PATH.name,
        drive_run / TEST768_PATH.name,
        drive_run / BLEND_PATH.name,
        drive_run / "run_metadata.json",
        drive_run / "final_adapter" / "adapter_model.safetensors",
    ]
    missing_drive_artifacts = [
        str(path) for path in required_drive_artifacts if not path.exists()
    ]
    assert not missing_drive_artifacts, (
        "Drive 최종 백업이 불완전하여 런타임을 유지합니다: "
        f"{missing_drive_artifacts}"
    )
    print(
        f"Drive 백업 검증 완료. {RELEASE_DELAY_SECONDS}초 후 GPU 런타임을 "
        "자동으로 연결 해제하고 삭제합니다."
    )
    sys.stdout.flush()
    time.sleep(RELEASE_DELAY_SECONDS)
    from google.colab import runtime
    runtime.unassign()
else:
    print("AUTO_RELEASE_RUNTIME=False: 런타임을 유지합니다.")
